# 📡 TM Forum RAG Assistant
### Grounded Q&A over publicly available TM Forum API specifications

**Project goal:** Help telecom analysts, solution architects, and developers answer TM Forum API and domain questions through a conversational interface.

**Final measured results**
- **5 public TM Forum specifications**
- **902 indexed chunks**
- **92% retrieval relevance**
- **92% answer faithfulness**
- **8.45 sec average end-to-end latency**

> This notebook is a compact technical demo. It shows how the architecture evolved without duplicating the full production codebase.


---
## 1. Knowledge Corpus

The corpus grew from 3 billing/payment-focused APIs to 5 APIs spanning customer, product ordering, account, billing, and payment domains.


### Pictorial view — From public specifications to searchable knowledge

<svg xmlns="http://www.w3.org/2000/svg" width="100%" viewBox="0 0 1200 300" role="img" aria-label="TM Forum ingestion flow">
  <defs>
    <linearGradient id="ingBg" x1="0" x2="1"><stop stop-color="#071A35"/><stop offset="1" stop-color="#102C55"/></linearGradient>
    <linearGradient id="ingCard" x1="0" x2="1"><stop stop-color="#173D70"/><stop offset="1" stop-color="#235C9E"/></linearGradient>
    <filter id="shadow" x="-20%" y="-20%" width="140%" height="150%"><feDropShadow dx="0" dy="5" stdDeviation="5" flood-color="#020B18" flood-opacity=".28"/></filter>
    <marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#7DB4FF"/></marker>
  </defs>
  <rect width="1200" height="300" rx="22" fill="url(#ingBg)"/>
  <text x="54" y="52" fill="#DCEBFF" font-family="Arial, sans-serif" font-size="22" font-weight="700">Offline, traceable ingestion pipeline</text>
  <text x="54" y="80" fill="#9DB6D5" font-family="Arial, sans-serif" font-size="14">Every chunk retains API and page metadata for later citation.</text>
  <g font-family="Arial, sans-serif" text-anchor="middle">
    <g filter="url(#shadow)"><rect x="55" y="125" width="188" height="104" rx="16" fill="url(#ingCard)"/><text x="149" y="165" fill="white" font-size="27">PDF</text><text x="149" y="195" fill="#DCEBFF" font-size="15" font-weight="700">Public TM Forum PDFs</text><text x="149" y="215" fill="#B8D4F4" font-size="12">5 API specifications</text></g>
    <path d="M253 177 H306" stroke="#7DB4FF" stroke-width="4" marker-end="url(#arrow)"/>
    <g filter="url(#shadow)"><rect x="319" y="125" width="168" height="104" rx="16" fill="#173D70"/><text x="403" y="165" fill="#7DB4FF" font-size="28">✦</text><text x="403" y="195" fill="white" font-size="15" font-weight="700">Cleaning</text><text x="403" y="215" fill="#B8D4F4" font-size="12">remove boilerplate</text></g>
    <path d="M497 177 H550" stroke="#7DB4FF" stroke-width="4" marker-end="url(#arrow)"/>
    <g filter="url(#shadow)"><rect x="563" y="125" width="204" height="104" rx="16" fill="#173D70"/><text x="665" y="162" fill="#7DB4FF" font-size="25">▤</text><text x="665" y="193" fill="white" font-size="15" font-weight="700">Chunking + Metadata</text><text x="665" y="215" fill="#B8D4F4" font-size="12">API · version · page · chunk ID</text></g>
    <path d="M777 177 H830" stroke="#7DB4FF" stroke-width="4" marker-end="url(#arrow)"/>
    <g filter="url(#shadow)"><rect x="843" y="125" width="145" height="104" rx="16" fill="#173D70"/><text x="915" y="165" fill="#7DB4FF" font-size="27">◈</text><text x="915" y="195" fill="white" font-size="15" font-weight="700">Embeddings</text><text x="915" y="215" fill="#B8D4F4" font-size="12">4096 dimensions</text></g>
    <path d="M998 177 H1050" stroke="#7DB4FF" stroke-width="4" marker-end="url(#arrow)"/>
    <g filter="url(#shadow)"><rect x="1063" y="125" width="104" height="104" rx="16" fill="#1F5B90"/><text x="1115" y="166" fill="white" font-size="20" font-weight="700">Pinecone</text><text x="1115" y="192" fill="#DCEBFF" font-size="13">+ BM25</text><text x="1115" y="215" fill="#B8D4F4" font-size="11">search indexes</text></g>
  </g>
</svg>


In [4]:
import pandas as pd

corpus = pd.DataFrame({
    "API": ["TMF666", "TMF622", "TMF678", "TMF676", "TMF629"],
    "Domain": [
        "Account Management",
        "Product Ordering Management",
        "Customer Bill Management",
        "Payment Management",
        "Customer Management",
    ],
    "Version": ["5.0.0", "5.0.0", "5.0.0", "4.0.0", "5.0.0"],
    "Chunks": [370, 299, 102, 66, 65],
})
corpus


,API,Domain,Version,Chunks
0,TMF666,Account Management,5.0.0,370
1,TMF622,Product Ordering Management,5.0.0,299
2,TMF678,Customer Bill Management,5.0.0,102
3,TMF676,Payment Management,4.0.0,66
4,TMF629,Customer Management,5.0.0,65


In [7]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.barh(corpus["API"], corpus["Chunks"])
plt.gca().invert_yaxis()
plt.xlabel("Indexed chunks")
plt.title("Knowledge Base Coverage — 902 Chunks")
for i, v in enumerate(corpus["Chunks"]):
    plt.text(v + 5, i, str(v), va="center")
plt.show()


ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']

**What we learned:** corpus composition matters. TMF666 and TMF622 dominate the knowledge base, while smaller specifications contribute focused domain coverage.


---
## 2. Inspect One Chunk

Each chunk keeps enough metadata to trace an answer back to the exact TM Forum specification, page, and chunk.

Run this notebook from the project root so it can read `data/processed/`.


In [3]:
import json
from pathlib import Path

path = Path("data/processed/TMF622_chunks.json")
chunks = json.loads(path.read_text(encoding="utf-8"))

sample = chunks[0]
{
    "chunk_id": sample.get("chunk_id"),
    "api_id": sample.get("api_id"),
    "api_name": sample.get("api_name"),
    "version": sample.get("version"),
    "page": sample.get("page"),
    "content_type": sample.get("content_type"),
    "text_preview": sample.get("text", "")[:500],
}


FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\TMF622_chunks.json'

A typical chunk preserves:

```text
api_id        → TMF622
api_name      → Product Ordering Management
version       → 5.0.0
page          → source PDF page
chunk_id      → TMF622_xxxx
content_type  → concept / field_definition / operation / json_example
```

This metadata later becomes part of the citation shown to the user.


---
## 3. Embedding

The production pipeline embeds every cleaned chunk using **Nebius `Qwen/Qwen3-Embedding-8B`**.

- Embedding dimension: **4096**
- Vector database: **Pinecone**
- Similarity metric: **cosine**

The demo cell below is disabled by default so opening the notebook does not make a paid API call.


In [ ]:
# Production pattern — enable only when you want to make an API call.

# from openai import OpenAI
# import os
#
# client = OpenAI(
#     base_url="https://api.tokenfactory.nebius.com/v1/",
#     api_key=os.environ["NEBIUS_API_KEY"],
# )
#
# response = client.embeddings.create(
#     model="Qwen/Qwen3-Embedding-8B",
#     input="TM Forum sample chunk",
# )
#
# vector = response.data[0].embedding
# print("Embedding dimension:", len(vector))


---
## 4. Dense Retrieval Baseline

The first version used only Pinecone semantic retrieval.

It usually found the correct API, but often ranked a **JSON example** above the **exact conceptual definition**.

**Observed example**

> `What is a BillingAccount?`

Dense retrieval returned multiple BillingAccount JSON examples before the concise concept definition.

**Lesson:** semantic similarity alone was not enough for dense technical documentation.


---
## 5. Add BM25 for Exact Technical Terms

BM25 improved exact matching for terms such as:

- `BillingAccount`
- `CustomerBill`
- `POST /payment`
- `Mandatory Attributes`

But keyword retrieval introduced a different issue: related resources can share nearly identical vocabulary.

For example, **Payment** and **Refund** both contain `totalAmount`, `paymentMethod`, and `account`.

**Lesson:** keyword retrieval improved exactness, but semantic distinctions were still needed.


---
## 6. Hybrid Retrieval + Reciprocal Rank Fusion

Instead of choosing between dense and BM25 retrieval, the system combines them.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
ax.axis("off")

nodes = {
    "User Question": (0.5, 0.92),
    "Dense Search
Pinecone": (0.28, 0.73),
    "BM25 Search
Keyword": (0.72, 0.73),
    "RRF Fusion": (0.5, 0.56),
    "LLM Reranking": (0.5, 0.42),
    "LangGraph
Evidence Check": (0.5, 0.27),
    "Grounded Answer": (0.28, 0.10),
    "Refusal": (0.72, 0.10),
}

for label, (x, y) in nodes.items():
    ax.text(x, y, label, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="black"))

arrows = [
    ((0.5,0.88),(0.28,0.78)), ((0.5,0.88),(0.72,0.78)),
    ((0.28,0.68),(0.5,0.60)), ((0.72,0.68),(0.5,0.60)),
    ((0.5,0.52),(0.5,0.46)), ((0.5,0.38),(0.5,0.32)),
    ((0.46,0.22),(0.30,0.14)), ((0.54,0.22),(0.70,0.14)),
]
for (x1,y1),(x2,y2) in arrows:
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle="->", lw=1.5))

plt.title("Final RAG Architecture")
plt.show()


**Why RRF?** Dense similarity scores and BM25 scores are not directly comparable. Reciprocal Rank Fusion combines **rank positions** instead of trying to normalize incompatible score scales.


---
## 7. LLM Reranking

Hybrid retrieval improved candidate coverage, but the best chunk was not always ranked first.

The LLM reranker receives the user question plus hybrid candidates and reorders them by direct relevance.

### Example improvement

For:

> **What is a ProductOrder?**

Raw hybrid retrieval ranked an API operation above the conceptual definition.

After reranking, `TMF622_0014 — ProductOrder resource` moved to **Rank #1**.

**Lesson:** retrieval recall and final relevance ranking are separate problems.


---
## 8. LangGraph Evidence Gate

A relevant-looking chunk is not automatically enough evidence to answer.


In [ ]:
state = {
    "question": "...",
    "chunk_ids": [],
    "context": "",
    "evidence_sufficient": False,
    "answer": "",
}
state


The graph follows two branches:

```text
Retrieve → Rerank → Evidence Check
                     ├── sufficient → grounded answer + citations
                     └── insufficient → refusal
```

This prevents the system from answering simply because retrieval found something vaguely related.


### Pictorial view — Evidence-first hybrid RAG architecture

<svg xmlns="http://www.w3.org/2000/svg" width="100%" viewBox="0 0 1200 510" role="img" aria-label="TM Forum hybrid RAG architecture">
  <defs>
    <linearGradient id="archBg" x1="0" x2="1"><stop stop-color="#071A35"/><stop offset="1" stop-color="#112E57"/></linearGradient>
    <linearGradient id="primary" x1="0" x2="1"><stop stop-color="#2463A5"/><stop offset="1" stop-color="#3D8ED6"/></linearGradient>
    <filter id="archShadow" x="-20%" y="-20%" width="140%" height="150%"><feDropShadow dx="0" dy="5" stdDeviation="5" flood-color="#020B18" flood-opacity=".3"/></filter>
    <marker id="archArrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#87BFFF"/></marker>
  </defs>
  <rect width="1200" height="510" rx="22" fill="url(#archBg)"/>
  <text x="52" y="52" fill="#E7F1FF" font-family="Arial, sans-serif" font-size="22" font-weight="700">TM Forum RAG Assistant — retrieval, ranking, and evidence validation</text>
  <text x="52" y="80" fill="#A9C2E0" font-family="Arial, sans-serif" font-size="14">Parallel retrieval maximizes coverage; the evidence gate protects answer quality.</text>
  <g font-family="Arial, sans-serif" text-anchor="middle">
    <g filter="url(#archShadow)"><rect x="63" y="188" width="145" height="92" rx="16" fill="url(#primary)"/><text x="135" y="225" fill="white" font-size="16" font-weight="700">User Question</text><text x="135" y="250" fill="#E7F1FF" font-size="12">telecom API query</text></g>
    <path d="M218 234 H270 V155 H322" fill="none" stroke="#87BFFF" stroke-width="4" marker-end="url(#archArrow)"/><path d="M218 234 H270 V334 H322" fill="none" stroke="#87BFFF" stroke-width="4" marker-end="url(#archArrow)"/>
    <g filter="url(#archShadow)"><rect x="334" y="110" width="180" height="90" rx="16" fill="#173D70"/><text x="424" y="148" fill="#87BFFF" font-size="17" font-weight="700">Dense Search</text><text x="424" y="172" fill="white" font-size="14">Pinecone</text><text x="424" y="189" fill="#B8D4F4" font-size="11">semantic similarity</text></g>
    <g filter="url(#archShadow)"><rect x="334" y="289" width="180" height="90" rx="16" fill="#173D70"/><text x="424" y="327" fill="#87BFFF" font-size="17" font-weight="700">BM25 Search</text><text x="424" y="351" fill="white" font-size="14">keyword retrieval</text><text x="424" y="368" fill="#B8D4F4" font-size="11">exact technical terms</text></g>
    <path d="M524 155 H575 V230 H627" fill="none" stroke="#87BFFF" stroke-width="4" marker-end="url(#archArrow)"/><path d="M524 334 H575 V258 H627" fill="none" stroke="#87BFFF" stroke-width="4" marker-end="url(#archArrow)"/>
    <g filter="url(#archShadow)"><rect x="639" y="198" width="142" height="92" rx="16" fill="#215C93"/><text x="710" y="235" fill="white" font-size="16" font-weight="700">RRF Fusion</text><text x="710" y="258" fill="#DCEBFF" font-size="12">merge ranked lists</text><text x="710" y="276" fill="#B8D4F4" font-size="11">better coverage</text></g>
    <path d="M791 244 H845" stroke="#87BFFF" stroke-width="4" marker-end="url(#archArrow)"/>
    <g filter="url(#archShadow)"><rect x="857" y="198" width="155" height="92" rx="16" fill="#276DA9"/><text x="934" y="235" fill="white" font-size="16" font-weight="700">LLM Reranking</text><text x="934" y="258" fill="#DCEBFF" font-size="12">select best evidence</text><text x="934" y="276" fill="#B8D4F4" font-size="11">question-aware order</text></g>
    <path d="M1022 244 H1070" stroke="#87BFFF" stroke-width="4" marker-end="url(#archArrow)"/>
    <g filter="url(#archShadow)"><rect x="1082" y="198" width="100" height="92" rx="16" fill="#2C7AB8"/><text x="1132" y="230" fill="white" font-size="14" font-weight="700">LangGraph</text><text x="1132" y="250" fill="white" font-size="14" font-weight="700">Evidence</text><text x="1132" y="270" fill="#DCEBFF" font-size="12">Check</text></g>
  </g>
  <path d="M1132 301 V355" stroke="#87BFFF" stroke-width="4" marker-end="url(#archArrow)"/>
  <g font-family="Arial, sans-serif"><text x="1086" y="331" fill="#7DE0B4" font-size="12" font-weight="700">SUFFICIENT</text><text x="1141" y="331" fill="#FFB8A8" font-size="12" font-weight="700">INSUFFICIENT</text></g>
  <g filter="url(#archShadow)" font-family="Arial, sans-serif" text-anchor="middle"><rect x="840" y="372" width="210" height="88" rx="16" fill="#1E7C65"/><text x="945" y="408" fill="white" font-size="16" font-weight="700">Grounded Answer</text><text x="945" y="432" fill="#D7FFF1" font-size="13">with citations</text><rect x="1067" y="372" width="105" height="88" rx="16" fill="#974A48"/><text x="1119" y="407" fill="white" font-size="15" font-weight="700">Refusal</text><text x="1119" y="432" fill="#FFE0DC" font-size="12">insufficient evidence</text></g>
</svg>


---
## 9. Grounded Answer Example

**Question**

> What fields are mandatory when creating a Payment?

**Grounded answer**

- `totalAmount`
- `paymentMethod`
- `account`

**Citation**

`[TMF676 | Page 37 | Chunk TMF676_0045]`

The generation prompt explicitly distinguishes mandatory fields, optional/example fields, patchable fields, example enum values, and complete enumerations.


---
## 10. Refusal Example

**Question**

> What is the SLA for resolving a payment dispute?

**Result**

```text
EVIDENCE SUFFICIENT: False

I could not find enough information in the retrieved TM Forum documentation.
```

This is an intentional success case: the system refuses instead of inventing an SLA.


---
## 11. Evaluation

The final system was evaluated on **25 questions** covering definitions, API operations, mandatory fields, multi-resource comparisons, cross-document relationships, updates, enumeration grounding, and unsupported questions.


In [ ]:
evaluation_metrics = pd.DataFrame({
    "Metric": [
        "Retrieval relevance",
        "Answer faithfulness",
        "Answer completeness",
        "Citation correctness",
    ],
    "Score": [92, 92, 88, 92],
})
evaluation_metrics


In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(evaluation_metrics["Metric"], evaluation_metrics["Score"])
plt.xlim(0, 100)
plt.xlabel("Percent")
plt.title("Manual Evaluation — 25 Questions")
for i, v in enumerate(evaluation_metrics["Score"]):
    plt.text(v + 1, i, f"{v}%", va="center")
plt.show()


### Main failure pattern

The remaining weakness is **multi-entity comparison retrieval**.

Examples:
- `CustomerBill` vs `CustomerBillOnDemand`
- `Payment` vs `Refund`

The retriever sometimes finds strong evidence for one entity but not enough balanced evidence for both. The evidence gate then refuses rather than forcing an answer.

**Future improvement:** retrieve each side of a comparison separately before fusion.


---
## 12. Latency

Five representative end-to-end queries were measured across the full pipeline:

`Hybrid retrieval → RRF → reranking → evidence validation → generation/refusal`


In [ ]:
latency = pd.DataFrame({
    "Query Type": ["Definition", "Operation", "Exact fields", "Cross-document", "Unsupported"],
    "Seconds": [10.29, 7.48, 5.68, 6.09, 12.68],
})
latency


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(latency["Query Type"], latency["Seconds"])
plt.ylabel("Seconds")
plt.title("End-to-End Latency Snapshot")
plt.xticks(rotation=20, ha="right")
for i, v in enumerate(latency["Seconds"]):
    plt.text(i, v + 0.25, f"{v:.2f}s", ha="center")
plt.show()


**Latency summary**

- Average: **8.45 sec**
- Median: **7.48 sec**
- Minimum: **5.68 sec**
- Maximum: **12.68 sec**

For this prototype, a practical target of **under 15 seconds end-to-end** was considered acceptable while prioritizing grounded answers.


---
## 13. The RAG Ladder — Complexity Was Earned

The project started with the simplest viable RAG and added each layer only after evaluation exposed a real failure.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.axis("off")

steps = [
    ("1", "Dense RAG", "Semantic baseline"),
    ("2", "BM25", "Exact terminology"),
    ("3", "Hybrid + RRF", "Lexical + semantic"),
    ("4", "LLM Reranker", "Final relevance"),
    ("5", "LangGraph Gate", "Answer or refuse"),
]
xs = [0.08, 0.27, 0.46, 0.65, 0.84]

for i, ((n, title, subtitle), x) in enumerate(zip(steps, xs)):
    ax.text(x, 0.55, f"{n}
{title}
{subtitle}", ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black"))
    if i < len(xs)-1:
        ax.annotate("", xy=(xs[i+1]-0.07,0.55), xytext=(x+0.07,0.55),
                    arrowprops=dict(arrowstyle="->", lw=1.5))

ax.text(0.5, 0.15,
        "Each layer was added only after testing exposed a specific failure mode.",
        ha="center")
plt.show()


### Why each rung was added

| Layer | Failure that justified it |
|---|---|
| Dense RAG | Semantic retrieval often returned examples instead of exact definitions |
| BM25 | Exact API terminology needed stronger lexical matching |
| Hybrid + RRF | Neither dense nor sparse retrieval was sufficient alone |
| LLM reranking | Relevant candidates existed but were not always ranked first |
| LangGraph evidence gate | Related evidence could still be insufficient to safely answer |

**Key principle:** do not add architectural complexity until evaluation demonstrates why it is needed.


### Pictorial view — The RAG ladder: complexity earned through evaluation

<svg xmlns="http://www.w3.org/2000/svg" width="100%" viewBox="0 0 1200 350" role="img" aria-label="RAG complexity ladder">
  <defs>
    <linearGradient id="ladBg" x1="0" x2="1"><stop stop-color="#071A35"/><stop offset="1" stop-color="#102C55"/></linearGradient>
    <linearGradient id="rung" x1="0" x2="1"><stop stop-color="#1E568D"/><stop offset="1" stop-color="#347FBC"/></linearGradient>
    <filter id="ladShadow" x="-20%" y="-20%" width="140%" height="150%"><feDropShadow dx="0" dy="5" stdDeviation="5" flood-color="#020B18" flood-opacity=".3"/></filter>
    <marker id="ladArrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#87BFFF"/></marker>
  </defs>
  <rect width="1200" height="350" rx="22" fill="url(#ladBg)"/>
  <text x="52" y="52" fill="#E7F1FF" font-family="Arial, sans-serif" font-size="22" font-weight="700">Each layer addresses a tested retrieval or grounding failure</text>
  <text x="52" y="80" fill="#A9C2E0" font-family="Arial, sans-serif" font-size="14">Move upward only when the previous approach exposes a meaningful gap.</text>
  <path d="M110 210 H1095" stroke="#5C91C7" stroke-width="7" stroke-linecap="round"/>
  <g font-family="Arial, sans-serif" text-anchor="middle">
    <g filter="url(#ladShadow)"><circle cx="145" cy="210" r="46" fill="url(#rung)"/><text x="145" y="216" fill="white" font-size="21" font-weight="700">1</text><text x="145" y="120" fill="white" font-size="16" font-weight="700">Dense RAG</text><text x="145" y="143" fill="#B8D4F4" font-size="12">semantic baseline</text><text x="145" y="293" fill="#FFB8A8" font-size="12" font-weight="700">Examples outrank</text><text x="145" y="310" fill="#FFB8A8" font-size="12" font-weight="700">definitions</text></g>
    <path d="M201 210 H313" stroke="#87BFFF" stroke-width="3" marker-end="url(#ladArrow)"/>
    <g filter="url(#ladShadow)"><circle cx="357" cy="210" r="46" fill="url(#rung)"/><text x="357" y="216" fill="white" font-size="21" font-weight="700">2</text><text x="357" y="120" fill="white" font-size="16" font-weight="700">BM25</text><text x="357" y="143" fill="#B8D4F4" font-size="12">exact terminology</text><text x="357" y="293" fill="#FFB8A8" font-size="12" font-weight="700">Exact-match alone</text><text x="357" y="310" fill="#FFB8A8" font-size="12" font-weight="700">misses intent</text></g>
    <path d="M413 210 H525" stroke="#87BFFF" stroke-width="3" marker-end="url(#ladArrow)"/>
    <g filter="url(#ladShadow)"><circle cx="569" cy="210" r="46" fill="url(#rung)"/><text x="569" y="216" fill="white" font-size="21" font-weight="700">3</text><text x="569" y="120" fill="white" font-size="16" font-weight="700">Hybrid + RRF</text><text x="569" y="143" fill="#B8D4F4" font-size="12">coverage across signals</text><text x="569" y="293" fill="#FFB8A8" font-size="12" font-weight="700">Best evidence is</text><text x="569" y="310" fill="#FFB8A8" font-size="12" font-weight="700">not always first</text></g>
    <path d="M625 210 H737" stroke="#87BFFF" stroke-width="3" marker-end="url(#ladArrow)"/>
    <g filter="url(#ladShadow)"><circle cx="781" cy="210" r="46" fill="url(#rung)"/><text x="781" y="216" fill="white" font-size="21" font-weight="700">4</text><text x="781" y="120" fill="white" font-size="16" font-weight="700">LLM Reranking</text><text x="781" y="143" fill="#B8D4F4" font-size="12">question-aware ranking</text><text x="781" y="293" fill="#FFB8A8" font-size="12" font-weight="700">Relevant context can</text><text x="781" y="310" fill="#FFB8A8" font-size="12" font-weight="700">still be insufficient</text></g>
    <path d="M837 210 H949" stroke="#87BFFF" stroke-width="3" marker-end="url(#ladArrow)"/>
    <g filter="url(#ladShadow)"><circle cx="993" cy="210" r="46" fill="#24806B"/><text x="993" y="216" fill="white" font-size="21" font-weight="700">5</text><text x="993" y="120" fill="white" font-size="16" font-weight="700">Evidence Gate</text><text x="993" y="143" fill="#B8E4D8" font-size="12">grounded answer or refusal</text><text x="993" y="293" fill="#92E4C7" font-size="12" font-weight="700">Stops unsupported</text><text x="993" y="310" fill="#92E4C7" font-size="12" font-weight="700">claims</text></g>
  </g>
</svg>


---
## 14. Final Architecture & Stack

| Layer | Technology |
|---|---|
| PDF ingestion | Python / LangChain loader |
| Chunking | Recursive text splitting + metadata |
| Embeddings | Nebius Qwen3-Embedding-8B |
| Vector database | Pinecone |
| Sparse retrieval | BM25 |
| Rank fusion | Reciprocal Rank Fusion |
| Reranking | LLM-based reranking |
| Workflow | LangGraph |
| Evaluation | Custom Python eval suite |
| UI | Streamlit |

### Final project statement

> **TM Forum RAG Assistant helps telecom professionals query five publicly available TM Forum API specifications through a conversational interface, achieving 92% retrieval relevance and 92% answer faithfulness across a 25-question evaluation.**


---
## 15. Next Improvements

Potential extensions:
- comparison-aware retrieval (`X` and `Y` retrieved separately),
- conversational memory with Mem0,
- automated document freshness/version checks,
- latency optimization,
- additional public TM Forum APIs,
- richer tracing/evaluation tooling.

The core lesson remains the same: **evaluation should drive architecture.**
